In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

D:\langchain_acedemy\lca-lc-foundations\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='google_genai:gemini-2.5-flash-lite',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='google_genai:gemini-2.5-flash-lite',
    tools=[square]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='google_genai:gemini-2.5-flash-lite',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [6]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='447f9c06-5ab4-464d-b327-26d84d00be57'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'call_subagent_1', 'arguments': '{"x": 456}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e54d3-881e-7b12-80a7-6b313135e111-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'c8150234-3ab8-452d-84da-8cf5e56c4eb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 20, 'total_tokens': 152, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='The square root of 456.0 is 21.354156504062622.', name='call_subagent_1', id='6e4ec3cd-bfa7-4144-9ab8-bc6ed5ae327c', tool_call_id='c8150234-3ab8-452d-84da-8cf5e56c4eb6'),
              AIMessage(content='The sq